# Cook County Property Value Estimation — Gradient Boosting
**Hands-on Machine Learning (2026–27)**  
**Author:** Vida Kayla BENSING  
**Model:** `HistGradientBoostingRegressor`

# 1. Intro

Our goal is to predict single-family home prices in Cook County, Illinois. Because the target is a continuous dollar amount, this is a supervised regression task.

### Benchmark & SOTA Context
We benchmark our models against the Cook County Assessor’s Office baseline (`ccao-data/model-res-avm`). The county previously relied on local linear models per township, but later adopted gradient-boosted trees (LightGBM) to better capture non-linear market behaviors—like diminishing returns on square footage and localized spatial clustering.

### Why Gradient Boosting?
Unlike Random Forests, which train trees in parallel to cut down variance via bagging, Gradient Boosting builds trees sequentially:
$$F_m(x) = F_{m-1}(x) + \eta h_m(x)$$
Each new tree $h_m$ fits the errors (pseudo-residuals) left behind by the previous trees. By using gradient descent with a learning rate, the algorithm progressively chips away at bias, which makes it particularly strong on structured tabular datasets.

### Metrics
We track three standard dollar-scale metrics:
- **MAE:** Average dollar miss per property ($\frac{1}{n}\sum \vert{}y_i - \hat{y}_i\vert{}$), keeping our evaluation robust to outliers.
- **RMSE:** Root mean squared error ($\sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}$), heavily penalizing large estimation mistakes.
- **$R^2$:** Proportion of price variance explained compared to a naive guess of the mean price ($1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$).

# 2. Preprocessing

### Timeline Split
Real estate prices drift over time due to inflation and shifting market conditions. Instead of a random train/test shuffle, we use a chronological split to test whether the model can predict future sales from past ones:
- **Train (2017–2023):** 276,685 homes
- **Validation (2024):** 30,545 homes (reserved strictly for tuning hyperparameter choices)
- **Test (2025):** 31,535 homes (untouched holdout for final scoring)

### Feature Handling
We retain 81 features (25 physical property specs and 56 neighborhood/spatial indicators). To avoid data leakage, our imputation and integer encodings (`make_tree_encoder()`) are calculated strictly from the 2017–2023 training data:
- Missing numerical values are filled with training medians
- Categorical features map unseen or missing entries to an explicit `'Missing'` label, which scikit-learn's histogram-based tree regressor handles natively.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor

from shared_preprocessing import load_splits, make_tree_encoder

# the splits
X_train, y_train, X_val, y_val, X_test, y_test = load_splits("../data/stage2_outputs")

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}   | y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

X_train shape: (276685, 81) | y_train shape: (276685,)
X_val shape:   (30545, 81)   | y_val shape:   (30545,)
X_test shape:  (31535, 81)  | y_test shape:  (31535,)


In [2]:
def evaluate_predictions(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    print(f"--- {name} ---")
    print(f"MAE:  ${mae:,.2f}")
    print(f"RMSE: ${rmse:,.2f}")
    print(f"R2:   {r2:.4f}\n")
    
    return {"model": name, "mae": mae, "rmse": rmse, "r2": r2}

# 3. Training & Validation Tuning

1. **Naive Mean Baseline:** Guesses the 2017–2023 training mean price ($\approx \$350\text{k}$) for every property, setting our baseline floor at $R^2 \approx 0$.
2. **Default HistGB:** Runs scikit-learn's default parameters as an initial benchmark.
3. **Hyperparameter Tuning:** We run a targeted grid search over learning rate ($\eta \in $), iteration count ($200 \text{ to } 300$), and tree depth ($8 \text{ to } 12$), selecting the combination with the lowest validation RMSE.

In [3]:
# simple mean baseline
train_mean = y_train.mean()
print(f"Train mean price: ${train_mean:,.2f}\n")

val_baseline_preds = np.full_like(y_val, fill_value=train_mean)
baseline_val_res = evaluate_predictions("Mean Baseline (Val 2024)", y_val, val_baseline_preds)

Train mean price: $350,713.99

--- Mean Baseline (Val 2024) ---
MAE:  $227,823.97
RMSE: $444,016.55
R2:   -0.0463



In [4]:
# initial HistGradientBoosting model

gb_pipe = make_pipeline(
    make_tree_encoder(),
    HistGradientBoostingRegressor(random_state=42)
)

gb_pipe.fit(X_train, y_train)

val_preds_default = gb_pipe.predict(X_val)
gb_default_res = evaluate_predictions("HistGB - Default (Val 2024)", y_val, val_preds_default)

--- HistGB - Default (Val 2024) ---
MAE:  $114,852.25
RMSE: $196,645.88
R2:   0.7948



In [5]:
# hyperparameter tuning (validation set)

tuning_configs = [
    {"learning_rate": 0.05, "max_iter": 200, "max_depth": 8},
    {"learning_rate": 0.1,  "max_iter": 250, "max_depth": 10},
    {"learning_rate": 0.08, "max_iter": 300, "max_depth": 12},
]

best_score = -float("inf")
best_model = None
best_params = None

for cfg in tuning_configs:
    pipe = make_pipeline(
        make_tree_encoder(),
        HistGradientBoostingRegressor(
            learning_rate=cfg["learning_rate"],
            max_iter=cfg["max_iter"],
            max_depth=cfg["max_depth"],
            random_state=42
        )
    )
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_val)
    r2 = r2_score(y_val, preds)
    mae = mean_absolute_error(y_val, preds)
    print(f"params: {cfg} -> Val R2: {r2:.4f}, MAE: ${mae:,.2f}")
    
    if r2 > best_score:
        best_score = r2
        best_model = pipe
        best_params = cfg

print(f"\nBest config: {best_params} (Val R2 = {best_score:.4f})")

params: {'learning_rate': 0.05, 'max_iter': 200, 'max_depth': 8} -> Val R2: 0.7931, MAE: $114,757.63
params: {'learning_rate': 0.1, 'max_iter': 250, 'max_depth': 10} -> Val R2: 0.8162, MAE: $107,526.22
params: {'learning_rate': 0.08, 'max_iter': 300, 'max_depth': 12} -> Val R2: 0.8142, MAE: $108,344.73

Best config: {'learning_rate': 0.1, 'max_iter': 250, 'max_depth': 10} (Val R2 = 0.8162)


In [6]:
tuning_df = pd.DataFrame(tuning_configs) 
tuning_df.to_csv("../results/gb_validation_tuning.csv", index=False)
print("Saved to ../results/gb_validation_tuning.csv")

Saved to ../results/gb_validation_tuning.csv


# 4. Evaluation

We lock in the best hyperparameters from the 2024 validation experiments and evaluate that single model on the unseen 2025 test set. We evaluate:
1. Overall dollar metrics against our naive baseline.
2. Prediction sanity (verifying that no homes receive impossible negative prices).
3. Performance by price bracket: Low-end ($< \$250\text{k}$), Mid-market ($\$250\text{k}\text{--}\$500\text{k}$ and $\$500\text{k}\text{--}\$1\text{M}$), and Luxury ($> \$1\text{M}$).

In [7]:
# final evaluation on 2025 test set
test_preds = best_model.predict(X_test)
gb_test_res = evaluate_predictions("HistGB (Test 2025)", y_test, test_preds)

test_baseline_preds = np.full_like(y_test, fill_value=train_mean)
baseline_test_res = evaluate_predictions("Mean Baseline (Test 2025)", y_test, test_baseline_preds)

--- HistGB (Test 2025) ---
MAE:  $125,945.35
RMSE: $260,565.96
R2:   0.7655

--- Mean Baseline (Test 2025) ---
MAE:  $249,127.97
RMSE: $551,842.59
R2:   -0.0518



In [8]:
import json

# calculate the metrics for the BEST model on the validation set
gb_val_res = evaluate_predictions("HistGB", y_val, best_model.predict(X_val))

final_metrics = pd.DataFrame([
    {"split": "Validation 2024", "model": "HistGB", "MAE": gb_val_res["mae"], "RMSE": gb_val_res["rmse"], "R2": gb_val_res["r2"]},
    {"split": "Test 2025", "model": "HistGB", "MAE": gb_test_res["mae"], "RMSE": gb_test_res["rmse"], "R2": gb_test_res["r2"]}
])

final_metrics.to_csv("../results/gb_metrics.csv", index=False)
print("Saved to ../results/gb_metrics.csv")

with open("../results/gb_config.json", "w") as f:
    json.dump(best_params, f, indent=4)
print("Saved to ../results/gb_config.json")

--- HistGB ---
MAE:  $107,526.22
RMSE: $186,088.72
R2:   0.8162

Saved to ../results/gb_metrics.csv
Saved to ../results/gb_config.json


### Takeaways from Generalization Metrics

- **Mean baseline failure ($R^2 = -0.0518$):** The naive baseline produces a negative score because 2025 market prices are higher than the 2017–2023 training average, making a static mean guess worse than simple variance.
- **Tuned model gains ($R^2 = 0.8162$ on 2024 validation):** The configuration with `lr=0.1`, `max_iter=250`, and `max_depth=10` performed best. Pushing tree depth to 12 slightly hurt validation performance ($R^2 = 0.8142$), indicating early signs of overfitting.
- **2025 Test generalization:** On completely unseen 2025 sales, HistGB achieves an $R^2$ of **0.7655** and an MAE of **$\$125,945$**, roughly halving the error of the mean baseline ($\approx \$249\text{k}$). The slight dip between 2024 and 2025 matches expected market drift over time.

In [9]:
# error analysis by price band

import pandas as pd
import numpy as np

# 1. create evaluation dataframe using 2025 test actuals and predictions
df_err = pd.DataFrame({
    "actual": y_test,
    "pred": test_preds
})

df_err["abs_error"] = (df_err["actual"] - df_err["pred"]).abs()
df_err["raw_error"] = df_err["pred"] - df_err["actual"]  # or actual - pred depending on convention

# 2. define the exact price bands used by Yuming
bins = [0, 250000, 500000, 1000000, np.inf]
labels = ["Up to 250k", "250k–500k", "500k–1M", "Above 1M"]
df_err["price_band"] = pd.cut(df_err["actual"], bins=bins, labels=labels)

# 3. aggregate MAE and Mean Error per band
gb_band_analysis = df_err.groupby("price_band", observed=False).agg(
    test_mae=("abs_error", "mean"),
    mean_error=("raw_error", "mean"),
    count=("actual", "count")
).reset_index()

# 4. check for negative price predictions
num_negative = (df_err["pred"] < 0).sum()
print(f"Negative price predictions in HistGB: {num_negative}")
display(gb_band_analysis)

Negative price predictions in HistGB: 0


,price_band,test_mae,mean_error,count
0,Up to 250k,54610.877104,-5218.655431,8694
1,250k–500k,92595.735549,-82769.966848,14348
2,500k–1M,174848.236459,-162282.519855,6070
3,Above 1M,456874.683155,-421124.564517,2423


In [10]:
import os

os.makedirs("../results", exist_ok=True)
gb_band_analysis.to_csv("../results/gb_errors_by_price_band.csv", index=False)
print("Saved to ../results/gb_errors_by_price_band.csv")

Saved to ../results/gb_errors_by_price_band.csv


### Takeaways from Error Distributions

- **Zero negative prices:** HistGB produced zero negative sale predictions across all 31,535 test properties. In comparison, Ridge regression predicted negative values for 214 properties, which is impossible in a real-world housing market.
- **Consistent gains over linear modeling:** Boosting cuts MAE across every price tier compared to Ridge—saving $\approx \$15\text{k}$ in average error on homes under $\$250\text{k}$ and $\approx \$148\text{k}$ on luxury homes over $\$1\text{M}$.
- **Underpredicting the luxury tail:** Both models struggle with homes over $\$1\text{M}$ (HistGB underpredicts by an average of $-\$421\text{k}$). Because luxury sales are rare in the training data, decision trees pull these extreme outliers toward the denser middle of the distribution.

# 5. Conclusion

### Summary
We tuned and tested `HistGradientBoostingRegressor` on roughly 338,000 Cook County property sales using a strict chronological split. 

### What Worked
- **Clear non-linear advantage:** HistGB achieved a 2025 test $R^2$ of **0.7655** (MAE of **$\$125,945$**), easily outperforming Ridge regression ($R^2 = 0.5817$).
- **Natural handling of coordinates:** By making threshold splits directly on latitude, longitude, and neighborhood codes, tree models isolate spatial price clusters without requiring manual coordinate transformations.
- **Bounded outputs:** Unlike unconstrained linear models, tree predictions stay within realistic price floors.

### What Didn't Work (Limitations)
- **High-end underprediction:** The sparse, heavy right tail of sales above $\$1\text{M}$ remains difficult to fit on an unscaled dollar loss, leading to severe underprediction on high-end homes.
- **Temporal decay:** Model performance drops from validation ($0.816$) to test ($0.765$), showing that property models decay over time as market dynamics evolve.

### What We Would Do With More Time
1. **Log Target Transformation:** Fit the model to $\log(\text{price})$ to convert absolute dollar losses into relative percentage errors, preventing luxury sales from skewing tree splits.
2. **Spatial Splines / Neighborhood Encodings:** Incorporate local spatial smoothing or target-encoded township boundaries to sharpen geographic price borders.
3. **Monotonic Constraints:** Enforce constraints so structural improvements (extra square footage or bathrooms) can never reduce estimated property values.